# KDIC Hybrid A 검색 평가기

`BGE-M3 Dense 85% + BM25-Nori Discard 15%`를 가중 RRF로 결합합니다.

- 각 검색기 후보: Top-10
- 가중 RRF 상수: 10
- 최종 평가 순위: Top-10
- 업무 필터: Gold 업무 사전 필터
- HCX API는 질문 Dense 임베딩에만 사용합니다.
- BM25-Nori와 RRF 계산에는 API 키가 필요하지 않습니다.

## 준비 파일

1. `KDIC_Hybrid_A_평가기.zip`
2. `KDIC_output.zip`
3. `Evaluation_DataSet_v4_1_SearchReady.xlsx`


In [ ]:
%pip -q install -U "numpy>=1.26,<3" "pandas==2.2.2" "openpyxl>=3.1,<4" "JPype1>=1.5,<2"

import getpass
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

WORK_ROOT=Path('/content/kdic_hybrid_a_evaluation')
EVALUATOR_ROOT=WORK_ROOT/'evaluator'
RESULT_ROOT=WORK_ROOT/'results'
WORK_ROOT.mkdir(parents=True,exist_ok=True)
print('환경 준비 완료:',WORK_ROOT)
print('Java 실행 파일:',shutil.which('java'))
if shutil.which('java') is None:
    raise RuntimeError('Java를 찾지 못했습니다. Colab 기본 런타임에서 다시 실행해 주세요.')


## 1. 파일 업로드

아래 셀을 실행하고 준비한 파일 3개를 한 번에 선택합니다.


In [ ]:
from google.colab import files
uploaded=files.upload()
for filename,content in uploaded.items():
    target=WORK_ROOT/filename
    target.write_bytes(content)
    print(f'업로드: {target.name} ({target.stat().st_size:,} bytes)')


In [ ]:
dataset_candidates=list(WORK_ROOT.glob('*.xlsx'))
kdic_candidates=[p for p in WORK_ROOT.glob('*.zip') if p.name=='KDIC_output.zip' or p.name.startswith('KDIC_output (')]
evaluator_candidates=[p for p in WORK_ROOT.glob('*.zip') if 'Hybrid' in p.name and '평가기' in p.name and '평가결과' not in p.name]
if not dataset_candidates:
    raise RuntimeError('평가 XLSX를 찾을 수 없습니다.')
if not kdic_candidates:
    raise RuntimeError('KDIC_output.zip을 찾을 수 없습니다.')
if not evaluator_candidates:
    raise RuntimeError('KDIC_Hybrid_A_평가기.zip을 찾을 수 없습니다.')
def choose_latest(items):
    return max(items,key=lambda p:p.stat().st_mtime)
DATASET_PATH=choose_latest(dataset_candidates)
KDIC_ZIP_PATH=choose_latest(kdic_candidates)
EVALUATOR_ZIP_PATH=choose_latest(evaluator_candidates)
if EVALUATOR_ROOT.exists():
    shutil.rmtree(EVALUATOR_ROOT)
EVALUATOR_ROOT.mkdir(parents=True)
with zipfile.ZipFile(EVALUATOR_ZIP_PATH) as archive:
    archive.extractall(EVALUATOR_ROOT)
scripts=list(EVALUATOR_ROOT.rglob('evaluate_hybrid_a.py'))
if len(scripts)!=1:
    raise RuntimeError(f'평가 스크립트를 하나로 결정할 수 없습니다: {scripts}')
EVALUATOR_SCRIPT=scripts[0]
print('평가데이터셋:',DATASET_PATH.name)
print('검색 데이터:',KDIC_ZIP_PATH.name)
print('평가기:',EVALUATOR_ZIP_PATH.name)


## 2. Hybrid A 조건

비교 결과에서 선정한 조건을 고정합니다.


In [ ]:
DENSE_WEIGHT=0.85
NORI_WEIGHT=0.15
RRF_CONSTANT=10
CANDIDATE_K=10
FINAL_K=10
BM25_K1=1.5
BM25_B=0.75
LUCENE_VERSION='9.12.2'
print({
    '검색 방식':'Hybrid A',
    'Dense':'BGE-M3',
    'Nori 모드':'discard',
    'Dense 가중치':DENSE_WEIGHT,
    'Nori 가중치':NORI_WEIGHT,
    'RRF 상수':RRF_CONSTANT,
    '후보 수':CANDIDATE_K,
    '최종 Top-K':FINAL_K,
})


## 3. Dry-run

API와 Nori 검색을 실행하지 않고 질문·청크·임베딩·Gold 연결을 검사합니다.


In [ ]:
dry_dir=WORK_ROOT/'results_dry'
cmd=[
    sys.executable,str(EVALUATOR_SCRIPT),
    '--dataset',str(DATASET_PATH),
    '--kdic-zip',str(KDIC_ZIP_PATH),
    '--output-dir',str(dry_dir),
    '--candidate-k',str(CANDIDATE_K),
    '--final-k',str(FINAL_K),
    '--dense-weight',str(DENSE_WEIGHT),
    '--nori-weight',str(NORI_WEIGHT),
    '--rrf-constant',str(RRF_CONSTANT),
    '--bm25-k1',str(BM25_K1),
    '--bm25-b',str(BM25_B),
    '--lucene-version',LUCENE_VERSION,
    '--dry-run',
]
subprocess.run(cmd,cwd=EVALUATOR_ROOT,check=True)


## 4. HCX API 키 불러오기

Colab Secrets에 `HCX_API_KEY`가 있으면 자동으로 사용합니다.  
Secret이 없거나 노트북 접근 권한이 꺼져 있을 때만 보안 입력창이 표시됩니다.


In [ ]:
from google.colab import userdata

try:
    HCX_API_KEY=(userdata.get('HCX_API_KEY') or '').strip()
except Exception:
    HCX_API_KEY=''

if HCX_API_KEY:
    print('Colab Secrets의 HCX_API_KEY를 불러왔습니다.')
else:
    print('HCX_API_KEY Secret을 읽지 못해 보안 입력창을 표시합니다.')
    HCX_API_KEY=getpass.getpass('HCX API 키를 입력하세요: ').strip()

if not HCX_API_KEY:
    raise RuntimeError('API 키가 비어 있습니다.')
if HCX_API_KEY.lower().startswith('bearer '):
    raise RuntimeError("API 키 앞에 'Bearer '를 붙이지 마세요.")
os.environ['HCX_API_KEY']=HCX_API_KEY
print('HCX_API_KEY가 현재 Colab 세션 환경 변수에 등록되었습니다.')


## 5. Hybrid A 전체 평가

질문 Dense 임베딩은 캐시되며, Nori와 RRF는 로컬에서 실행됩니다.


In [ ]:
RESULT_ROOT.mkdir(parents=True,exist_ok=True)
cmd=[
    sys.executable,str(EVALUATOR_SCRIPT),
    '--dataset',str(DATASET_PATH),
    '--kdic-zip',str(KDIC_ZIP_PATH),
    '--output-dir',str(RESULT_ROOT),
    '--candidate-k',str(CANDIDATE_K),
    '--final-k',str(FINAL_K),
    '--dense-weight',str(DENSE_WEIGHT),
    '--nori-weight',str(NORI_WEIGHT),
    '--rrf-constant',str(RRF_CONSTANT),
    '--bm25-k1',str(BM25_K1),
    '--bm25-b',str(BM25_B),
    '--lucene-version',LUCENE_VERSION,
]
subprocess.run(cmd,cwd=EVALUATOR_ROOT,check=True)


## 6. 평가 결과 확인


In [ ]:
summary=json.loads((RESULT_ROOT/'summary.json').read_text(encoding='utf-8'))
overall=pd.DataFrame([summary['overall']])
by_domain=pd.read_csv(RESULT_ROOT/'summary_by_domain.csv')
questions=pd.read_csv(RESULT_ROOT/'question_results.csv')
print('검색 방식:',summary['retriever'])
print('전체 평가 결과')
display(overall)
print('도메인별 평가 결과')
display(by_domain)
print('Hit@3 실패 질문 예시')
display(questions.loc[
    questions['hit_at_3']==0,
    ['evaluation_id','question','domain','gold_chunk_ids','retrieved_chunk_ids']
].head(10))


## 7. 결과 다운로드

다운로드한 ZIP을 KDIC 검색 품질 비교 대시보드에 추가합니다.


In [ ]:
from google.colab import files
archive_path=shutil.make_archive(
    '/content/KDIC_Hybrid_A_평가결과',
    'zip',
    RESULT_ROOT,
)
print('결과 압축:',archive_path)
files.download(archive_path)
